In [1]:
# --- Cell 1: accuracy, bias and tracking signal per product -----------------------------
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest

test = pd.read_csv("outputs/forecast_backtest.csv", parse_dates=["date"])
test["error"] = test["fc_blend"] - test["ordered_units"]       # + = over-forecast, - = under-forecast
working_days = test[test["date"].dt.dayofweek != 4]             # leave out Fridays (holiday)

acc = test.groupby(["product", "product_group"]).agg(actual_units=("ordered_units", "sum"),
                                                     forecast_units=("fc_blend", "sum")).reset_index()

# WAPE on weekly totals, because production is planned by the week (Thursday-Wednesday weeks)
weekly = test.groupby(["product", test["date"].dt.to_period("W-WED")])[["ordered_units", "fc_blend"]].sum()
weekly_error = (weekly["fc_blend"] - weekly["ordered_units"]).abs().groupby("product").sum()
acc["wape_pct"] = (weekly_error / weekly["ordered_units"].groupby("product").sum() * 100).values
acc["bias_pct"] = (acc["forecast_units"] - acc["actual_units"]) / acc["actual_units"] * 100

# Tracking signal = sum of errors / average size of an error.
# Beyond +/-4 means the forecast is wrong in the same direction day after day.
acc["tracking_signal"] = working_days.groupby("product")["error"].apply(lambda e: e.sum() / e.abs().mean()).values

# Flag only what a planner should act on: consistent AND large drift, or a poor forecast
acc["flag"] = "OK"
acc.loc[acc["wape_pct"] > 50, "flag"] = "Low accuracy"
acc.loc[(acc["tracking_signal"].abs() > 4) & (acc["bias_pct"].abs() > 20), "flag"] = "Drifting - review"

acc = acc.sort_values("actual_units", ascending=False).round(1)
acc.to_csv("outputs/forecast_accuracy_by_product.csv", index=False)
print(acc["flag"].value_counts())
acc.head(10)

flag
Drifting - review    18
OK                   13
Name: count, dtype: int64


/var/folders/46/jh4c0gp573ggq19nrvdvy1880000gn/T/ipykernel_51750/2120489707.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  working_days = test[test["date"].dt.dayofweek != 4]             # leave out Fridays (holiday)


,product,product_group,actual_units,forecast_units,wape_pct,bias_pct,tracking_signal,flag
24,SOS001L12P,S,142827.0,148661.6,16.9,4.1,2.3,OK
27,SOS005L04P,S,121284.0,150222.8,23.9,23.9,14.2,Drifting - review
1,ATN01K24P,A,92880.0,78080.8,34.9,-15.9,-8.6,OK
25,SOS002L09P,S,92698.0,98704.0,23.1,6.5,3.5,OK
30,SOS500M24P,S,52121.0,53737.7,15.8,3.1,2.4,OK
17,POV001L24P,P,45766.0,43177.8,29.3,-5.7,-3.3,OK
20,POV500M24P,P,24549.0,25634.1,28.2,4.4,2.2,OK
11,POP001L12P,P,22808.0,14257.7,38.4,-37.5,-10.3,Drifting - review
26,SOS003L04P,S,19185.0,26067.7,35.9,35.9,17.5,Drifting - review
2,ATN02K12P,A,17935.0,14167.3,21.0,-21.0,-10.6,Drifting - review


In [2]:
# --- Cell 2: AI anomaly detection with an Isolation Forest --------------------------------
# For each product, the model learns what normal daily orders look like and flags the oddest 2% of days.
flows = pd.read_parquet("data/processed/daily_flows.parquet")
flows["date"] = pd.to_datetime(flows["date"])
flows = flows[flows["product"].isin(acc["product"]) & (flows["date"].dt.dayofweek != 4)]

anomalies = []
for product, days in flows.groupby("product"):
    days = days.sort_values("date").copy()
    days["normal_level"] = days["ordered_units"].rolling(28, min_periods=7).mean().shift(1)  # last 4 weeks' average
    days = days.dropna(subset=["normal_level"])
    days["excess_units"] = days["ordered_units"] - days["normal_level"]                      # size of the surprise
    detector = IsolationForest(contamination=0.02, random_state=42)
    days["anomaly"] = detector.fit_predict(days[["ordered_units", "excess_units"]]) == -1
    anomalies.append(days[days["anomaly"]])

anomalies = pd.concat(anomalies)
anomalies["type"] = np.where(anomalies["excess_units"] > 0, "Spike", "Drop")
anomalies = anomalies.sort_values("excess_units", key=abs, ascending=False)   # biggest surprises first
anomalies = anomalies[["date", "product", "product_group", "ordered_units", "excess_units", "type"]]
anomalies["excess_units"] = anomalies["excess_units"].round()
anomalies.to_csv("outputs/demand_anomalies.csv", index=False)
print("Unusual order days:", len(anomalies))
anomalies.head(10)

/var/folders/46/jh4c0gp573ggq19nrvdvy1880000gn/T/ipykernel_51750/1905641977.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  flows = flows[flows["product"].isin(acc["product"]) & (flows["date"].dt.dayofweek != 4)]


Unusual order days: 124


,date,product,product_group,ordered_units,excess_units,type
8221,2023-02-14,SOS005L04P,S,29251.0,19645.0,Spike
7558,2023-02-14,SOS001L12P,S,28339.0,17317.0,Spike
8244,2023-03-09,SOS005L04P,S,28774.0,16403.0,Spike
8212,2023-02-05,SOS005L04P,S,24629.0,16320.0,Spike
7549,2023-02-05,SOS001L12P,S,25394.0,14983.0,Spike
7812,2023-03-19,SOS002L09P,S,20881.0,14547.0,Spike
8247,2023-03-12,SOS005L04P,S,25024.0,12909.0,Spike
7527,2023-01-14,SOS001L12P,S,1508.0,-12325.0,Drop
7584,2023-03-12,SOS001L12P,S,23725.0,12080.0,Spike
3567,2023-02-01,POP001L12P,P,12016.0,10868.0,Spike


In [3]:
# --- Cell 3: evidence for the root causes of the biggest misses --------------------------
biggest_misses = acc.reindex(acc["bias_pct"].abs().sort_values(ascending=False).index)
biggest_misses = biggest_misses[biggest_misses["actual_units"] > 20000].head(3)   # only products that matter
for _, row in biggest_misses.iterrows():
    weekly = flows[flows["product"] == row["product"]].set_index("date")["ordered_units"].resample("W-WED").sum()
    print(f"{row['product']}: bias {row['bias_pct']:+.0f}%  | last 10 weeks of orders: {weekly.tail(10).astype(int).tolist()}")

POP001L12P: bias -38%  | last 10 weeks of orders: [868, 3790, 4873, 3326, 3234, 4844, 4564, 9108, 5685, 3451]
SOS005L04P: bias +24%  | last 10 weeks of orders: [40985, 45055, 49133, 40701, 22053, 44354, 36294, 29035, 32906, 23049]
ATN01K24P: bias -16%  | last 10 weeks of orders: [11411, 12825, 28688, 17248, 18107, 26615, 27436, 35411, 20113, 9920]
